In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
!pip install torch
!pip install transformers[torch]
!pip install accelerate -U

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl (176.2 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (99 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21

In [ ]:
# NEEDS TO BE PARENT DIRECTORY OF TRAINING
dir= '/content/drive/MyDrive/BridgeAthletics/'
sub_dir='/Training/Data1/'
data_sub_dir='/Dataset1_reps'

In [ ]:
import json
import os
from tqdm import tqdm
import sys
import torch
import accelerate
from torch.utils.data import Dataset, DataLoader
sys.path.append(dir)
import pandas as pd
import numpy as np

#CUSTOM FUNCTIONS FROM FUNCTIONS.PY
from Training.Functions import *

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Using CPU")

Using CPU


## MODEL + TOKENIZER LIBRARY

In [ ]:
#MODEL SELECTION: GPT2
from transformers import GPT2LMHeadModel, GPT2Tokenizer, DataCollatorForLanguageModeling

In [ ]:
#MODEL SELECTION: T5
from transformers import T5Tokenizer, T5ForConditionalGeneration, DataCollatorForSeq2Seq

## DATASET CLASS - FORMATTING + PREPARING FOR TRAINING

In [ ]:
class InstructionDataset_GPT2(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        self.instruction_lengths = []
        for item in data:
            instruction_plus_input = format_model_input(item)
            response_text = f"\n\n### Response:\n{item['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )
            instruction_length = len(tokenizer.encode(instruction_plus_input))
            self.instruction_lengths.append(instruction_length)

    def __getitem__(self, index):
        # return self.instruction_lengths[index], self.encoded_texts[index] #(TO USE WITH CUSTOM COLLATE)
        return self.encoded_texts[index] #(TO USE WITH TRANSFORMERS COLLATE)

    def __len__(self):
        return len(self.data)

class InstructionDataset_T5(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.inputs=[]
        self.labels=[]
        for item in data:
            instruction_plus_input = format_model_input(item)
            response_text = f"\n\n### Response:\n{item['output']}"

            input_ids = tokenizer.encode(instruction_plus_input)
            label_ids = tokenizer.encode(response_text)

            self.inputs.append(input_ids)
            self.labels.append(label_ids)

    def __getitem__(self, index):
      return {
            'input_ids': torch.tensor(self.inputs[index], dtype=torch.long),
            'labels': torch.tensor(self.labels[index], dtype=torch.long)
        }

    def __len__(self):
        return len(self.data)

## CUSTOM COLLATE FUNCTION IF NEEDED

In [ ]:
def collated_fromMLMtoCLM(labels,instr_len):
    labels = labels[:,1:]
    new_labels = torch.zeros((labels.size(0), labels.size(1) + 1), dtype=labels.dtype)
    for i in range(0,len(labels)):
        if len(labels[i,:])==0:
            row_list=[end_of_text_token_id]

        else:
            if labels[i,-1]!=-100:
                row_list = labels[i].tolist()
                row_list.append(end_of_text_token_id)

            else:
                if (labels[i]==-100).all():
                    row_list = labels[i].tolist()
                    row_list.insert(0,end_of_text_token_id)
                else:
                    for j in range (len(labels[i,:])):
                        if labels[i,j+1]==-100:
                           row_list = labels[i].tolist()
                           row_list.insert(j+1,end_of_text_token_id)
                           break

        new_labels[i] = torch.tensor(row_list, dtype=labels.dtype).to(device)
        #new_labels[i,:instr_len[i]-1] = -100 #UNCOMMENT FOR INSTRUCTION MASKING IN LOSS FUNCTION

    return new_labels


In [ ]:
def CLM_Collator(tokenized_data_input_tuple, tokenizer=GPT2Tokenizer.from_pretrained('gpt2',padding_side="right", add_eos_token=True, add_bos_token=False)):
    tokenizer.pad_token = tokenizer.eos_token
    tokenized_data_input = [item[1] for item in tokenized_data_input_tuple]
    instr_lengths = [item[0] for item in tokenized_data_input_tuple]
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    collated_samples = data_collator(tokenized_data_input)
    collated_samples['labels'] = collated_fromMLMtoCLM(collated_samples['labels'],instr_lengths)
    return collated_samples

# def CLM_Collator(tokenized_data_input, tokenizer=GPT2Tokenizer.from_pretrained('gpt2',padding_side="right", add_eos_token=True, add_bos_token=False)):
#     tokenizer.pad_token = tokenizer.eos_token
#     data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
#     collated_samples=data_collator(tokenized_data_input)
#     collated_samples['labels'] = collated_fromMLMtoCLM(collated_samples['labels'])
#     return collated_samples

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## TEST FUNCTIONS (DATASET-BATCHES-COLLATE)

In [ ]:
def check_input_label_shapes(train_loader):
  for pairs in train_loader:
      print(pairs['input_ids'].shape, pairs['labels'].shape)

  print(pairs['input_ids'][0])
  print(pairs['labels'][0])

In [ ]:
def collator_decoder_test(train_loader):
  for j in train_loader:
      tensor = j['input_ids'][0]
      filtered_tensor = tensor[tensor != -100]
      token_ids = filtered_tensor.tolist()
      decoded_string = tokenizer.decode(token_ids, skip_special_tokens=False)
      print("Decoded Input:")
      print(decoded_string)

      tensor = j['labels'][0]
      filtered_tensor = tensor[tensor != -100]
      token_ids = filtered_tensor.tolist()
      decoded_string = tokenizer.decode(token_ids, skip_special_tokens=False)
      print("\nDecode Label:")
      print(decoded_string)
      break

## DOWNLOADING DATA + TRAIN_TEST_VAL SPLIT

In [ ]:
data = download_data(dir+data_sub_dir+'/finaldataset_shortblocks.json')
data = remove_extra_quotes(data)

print("Number Of Samples:",len(data),"\n")
print("Initial Sample Example:\n",data[0],"\n")

Number Of Samples: 1310 

Initial Sample Example:
 {'input': 'Warm Up Complex', 'output': [{'exercise': 'Barbell Front Squat', 'sets': 2, 'reps': [10, 10]}, {'exercise': 'Barbell RDL', 'sets': 2, 'reps': [10, 10]}]} 



In [ ]:
train_data,test_data,val_data = train_test_val_split(data)

In [ ]:
print("Training set length:", len(train_data), "//Validation set length:", len(val_data),"//Test set length:", len(test_data))

Training set length: 1113 //Validation set length: 66 //Test set length: 131


## BASE MODEL + TOKENIZER DOWNLOAD (RUN ONE CELL ONLY)

GPT2:

In [ ]:
#BASE GPT2 MODEL FROM HUGGING FACE
original_model_name="gpt2-medium"
model = GPT2LMHeadModel.from_pretrained(original_model_name)

#TOKENIZER + COLLATOR
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
end_of_text_token_id = tokenizer.encode("<|endoftext|>")[0]
tokenizer.pad_token = tokenizer.eos_token
data_collator_fn = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [ ]:
#SAVED GPT2 MODEL POST FINE-TUNING (IF EXISTS)
original_model_name="gpt2-medium"
model_name=dir+sub_dir+'/final_model_'+original_model_name
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
end_of_text_token_id = tokenizer.encode("<|endoftext|>")[0]
tokenizer.pad_token = tokenizer.eos_token
data_collator_fn = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

T5:

In [ ]:
#BASE T5 MODEL
original_model_name = "t5-base"
model = T5ForConditionalGeneration.from_pretrained(original_model_name)
tokenizer = T5Tokenizer.from_pretrained(original_model_name)
tokenizer.pad_token = tokenizer.eos_token
data_collator_fn = DataCollatorForSeq2Seq(tokenizer, model=model)

special_tokens_dict = {'additional_special_tokens': ['{', '}']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
model.resize_token_embeddings(len(tokenizer))


In [ ]:
#SAVED T5 MODEL POST FINE-TUNING (IF EXISTS)
original_model_name = "t5-base"
model_name = dir+sub_dir+'/final_model_'+original_model_name
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
data_collator_fn = DataCollatorForSeq2Seq(tokenizer, model=model)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
print(f"Num of param for {model_name}:",sum(p.numel() for p in model.parameters()))
print(f"Max Length: {model.config.n_positions}")
model.to(device)  # Move the model to the appropriate device

Num of param for /content/drive/MyDrive/BridgeAthletics//Training/final_model_t5-base: 222883584
Max Length: 512


T5ForConditionalGeneration(
  (shared): Embedding(32102, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32102, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

## TRAINING + TOKENIZER + INSTRUCTION-DATASET + COLLATOR INITALIZATION FOR TRAINING



In [ ]:
#TRAINING AND DATA SETTINGS

#GPT2
if "gpt2" in model_name:
  num_workers = 0
  batch_size = 8
  epochs=9

  torch.manual_seed(123)

  train_dataset = InstructionDataset_GPT2(train_data, tokenizer)
  val_dataset = InstructionDataset_GPT2(val_data, tokenizer)
  test_dataset = InstructionDataset_GPT2(test_data, tokenizer)

  #FOR TESTING PURPOSES:
  train_loader = DataLoader(train_dataset,batch_size=batch_size,collate_fn=data_collator_fn,
      shuffle=True,
      drop_last=True,
      num_workers=num_workers
  )

elif "t5" in model_name:
  #T5
  num_workers = 0
  batch_size = 8
  epochs=9

  torch.manual_seed(123)

  train_dataset = InstructionDataset_T5(train_data, tokenizer)
  val_dataset = InstructionDataset_T5(val_data, tokenizer)
  test_dataset = InstructionDataset_T5(test_data, tokenizer)

  #FOR TESTING PURPOSES:
  train_loader = DataLoader(train_dataset,batch_size=batch_size,collate_fn=data_collator_fn,
      shuffle=True,
      drop_last=True,
      num_workers=num_workers
  )

else:
  sys.exit("Error: model not defined")

In [ ]:
check_input_label_shapes(train_loader)
print('\n\n\n')
collator_decoder_test(train_loader)

## TRAINING HYPERPARAMS

In [ ]:
batch_size = 8
epochs=12
training_args = TrainingArguments(
    output_dir=(dir+sub_dir+'results_'+original_model_name),
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=2*batch_size,
    warmup_steps=int(0.1* epochs* (len(train_dataset)//batch_size)),
    weight_decay=0.1,
    logging_dir=dir+sub_dir+'logs_'+original_model_name,
    logging_steps=100,
    do_train=True,
    do_eval=True,
    eval_strategy="steps",
    eval_steps=int(len(train_dataset)/10),
    save_strategy="steps",
    save_steps=2*int(len(train_dataset)/10),
    save_total_limit=3,
    load_best_model_at_end=True,
    resume_from_checkpoint=True,
    lr_scheduler_type='linear',
    gradient_accumulation_steps=2,
    max_grad_norm=1.0,
    learning_rate=7.5e-5,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset= val_dataset,
    data_collator=data_collator_fn,
)

## TRAINING + EVAL

### GPT2

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
111,1.275500,0.475806
222,0.448600,0.402460
333,0.361500,0.376064
444,0.315300,0.381498
555,0.278900,0.366704
666,0.250600,0.377596
777,0.234900,0.381221


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=840, training_loss=0.4128862153916132, metrics={'train_runtime': 1487.565, 'train_samples_per_second': 8.978, 'train_steps_per_second': 0.565, 'total_flos': 3243102635507712.0, 'train_loss': 0.4128862153916132, 'epoch': 12.0})

In [ ]:
trainer.evaluate()

#TODO
trainer.evaluate(eval_dataset=test_dataset)

{'eval_loss': 0.3775961101055145,
 'eval_runtime': 2.2901,
 'eval_samples_per_second': 28.819,
 'eval_steps_per_second': 2.183,
 'epoch': 12.0}

### T5

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
111,0.237600,0.240960
222,0.234600,0.244300
333,0.228000,0.234225
444,0.217500,0.231219
555,0.210500,0.227054
666,0.199200,0.227166
777,0.196500,0.223905


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=840, training_loss=0.21330056020191737, metrics={'train_runtime': 802.8418, 'train_samples_per_second': 16.636, 'train_steps_per_second': 1.046, 'total_flos': 546464776412160.0, 'train_loss': 0.21330056020191737, 'epoch': 12.0})

In [ ]:
trainer.evaluate(eval_dataset=test_dataset)

{'eval_loss': 0.26872846484184265,
 'eval_runtime': 2.3173,
 'eval_samples_per_second': 56.531,
 'eval_steps_per_second': 3.884,
 'epoch': 12.0}

# TESTING : CHECKING MODEL OUTPUT EXAMPLES AND STORING THEM

### GPT2 FINETUNED OUTPUT EXAMPLES

In [ ]:
model.eval()
model_outputs=[]
data_to_use = train_data

for i in tqdm(range(len(data_to_use))):
  in_test = data_to_use[i]
  sample_out = data_to_use[i]['output']
  in_test=format_model_input(in_test)
  input_ids = tokenizer.encode(in_test, return_tensors="pt").to(device)
  # output = model.generate(
  #     input_ids=input_ids,
  #     eos_token_id=50256,
  #     max_length=len(input_ids) + 200,
  #     num_return_sequences=1,
  #     early_stopping=True,
  #     pad_token_id=50256,
  # )
  output = model.generate(
      input_ids=input_ids,
      eos_token_id=50256,
      max_length=len(input_ids) + 200,
      num_beams=3,
      num_return_sequences=1,
      early_stopping=True,
      pad_token_id=50256,
      #repetition_penalty=1.5,  #TO EXPERIMENT WITH
  )

  decoded_output=tokenizer.decode(output[0], skip_special_tokens=True)

  stop_sequence = "### Response"
  stop_index = decoded_output.find(stop_sequence, decoded_output.find(stop_sequence) + len(stop_sequence))
  if stop_index != -1:
      trimmed_output = decoded_output[:stop_index]
  else:
      trimmed_output = decoded_output

  print('expected output:',sample_out,'model output:',trimmed_output,'\n',sep='\n')
  model_outputs.append(trimmed_output)

 50%|█████     | 1/2 [01:06<01:06, 66.71s/it]

expected output:
[{'exercise': '1-Leg DB SLDL', 'sets': 3, 'reps': [8, 8, 8]}]
model output:
### Instruction:
Generate a workout block in JSON format for the following block title.

### Input:
full body strength

### Response:
[{'exercise': 'Barbell Power Clean','sets': 5,'reps': [3, 3, 3, 3, 3]}]






100%|██████████| 2/2 [02:10<00:00, 65.07s/it]

expected output:
[{'exercise': 'Barbell Bent-Over Row', 'sets': 3, 'reps': [8, 8, 8]}]
model output:
### Instruction:
Generate a workout block in JSON format for the following block title.

### Input:
cardio warmup

### Response:
[{'exercise': 'Foam Roll - Quads','sets': 1,'reps': [0]}, {'exercise': 'Foam Roll - IT Band','sets': 1,'reps': [0]}]}, {'exercise': 'Foam Roll - Hamstring','sets': 1,'reps': [0]}]






### T5 FINETUNED OUTPUT EXAMPLES

In [ ]:
model.eval()
model_outputs=[]
data_to_use = val_data

for i in tqdm(range(len(data_to_use))):
  in_test = data_to_use[i]
  sample_out = data_to_use[i]['output']
  in_test=format_model_input(in_test)
  print("in_test:\n",in_test) #UNCOMMENT TO PRINT
  input_ids = tokenizer.encode(in_test, return_tensors="pt").to(device)
  output = model.generate(
      input_ids=input_ids,
      eos_token_id=1,
      max_length=len(input_ids) + 200,
      num_beams=3,
      num_return_sequences=1,
      early_stopping=True,
      repetition_penalty=1.5,  #TO EXPERIMENT WITH
  )

  decoded_output=tokenizer.decode(output[0], skip_special_tokens=False)
  stop_sequence = "### Response"
  stop_index = decoded_output.find(stop_sequence, decoded_output.find(stop_sequence) + len(stop_sequence))
  if stop_index != -1:
      trimmed_output = decoded_output[:stop_index]
  else:
      trimmed_output = decoded_output

  print('expected output:',sample_out,'\nmodel output:',trimmed_output,'\n',sep='\n') #UNCOMMENT TO SEE OUTPUT
  model_outputs.append(trimmed_output)

  0%|          | 0/2 [00:00<?, ?it/s]

in_test:
 ### Instruction:
Generate a workout block in JSON format for the following block title.

### Input:
full body strength


 50%|█████     | 1/2 [00:06<00:06,  6.70s/it]

expected output:
[{'exercise': '1-Leg DB SLDL', 'sets': 3, 'reps': [8, 8, 8]}]

model output:
<pad> ### Response: [ { 'exercise': 'Barbell Front Squat','sets': 6,'reps': [3, 3, 3, 3, 3, 3] } ]</s>


in_test:
 ### Instruction:
Generate a workout block in JSON format for the following block title.

### Input:
cardio warmup


100%|██████████| 2/2 [00:22<00:00, 11.26s/it]

expected output:
[{'exercise': 'Barbell Bent-Over Row', 'sets': 3, 'reps': [8, 8, 8]}]

model output:
<pad> ### Response: [ { 'exercise': 'Squat','sets': 1,'reps': [10] }, { 'exercise': 'Front Lunge','sets': 1,'reps': [10] } ]</s>




### SAVE MODEL OUTPUTS

In [ ]:
if len(model_outputs)==len(train_data):
  output_file=dir+sub_dir+'final_model_'+original_model_name+'_model_outputs_train.json'

elif len(model_outputs)==len(test_data):
  output_file=dir+sub_dir+'final_model_'+original_model_name+'_model_outputs_test_with_rep_pen.json'

elif len(model_outputs)==len(val_data):
  output_file=dir+sub_dir+'final_model_'+original_model_name+'_model_outputs_val_with_rep_pen.json'

else:
  sys.exit("Error: lengths do not match with original dataset")

with open(output_file, 'w') as f:
  json.dump(model_outputs, f)

## OUTPUT CONVERSION TO LIST of DICTS + RUN TESTS + DATA ANALYSIS

In [ ]:
import ast

### CHOOSE MODEL AND DATASET OUTPUTS.
###!!!!!!DO NOT RUN FIRST CELL IF YOU WANT THE VALUES FROM "{MODEL}FINETUNED OUTPUT EXAMPLES" EXECUTION!!!!!!

In [ ]:
###MODEL OUTPUTS
file_to_read = '/content/drive/MyDrive/BridgeAthletics/Training/final_model_gpt2-medium_model_outputs_train.json'
with open(file_to_read, 'r') as f:
    model_outputs = json.load(f)

In [ ]:
data_outputs = [train_data[i]['output'] for i in range(len(train_data))]
main_dataset_df = model_output_list_to_df(data_outputs)
main_dataset_df

,block,exercise,sets,reps
0,0,Hex Bar Deadlift,3,"[10, 10, 10]"
1,0,SB Crunch,3,"[20, 20, 20]"
2,0,Front Bridge,3,"[0, 0, 0]"
3,0,Back Bridge,3,"[10, 10, 0]"
4,1,Banded Clamshell,2,"[10, 10]"
...,...,...,...,...
1868,1111,Standing Barbell Military Press (Front),1,[8]
1869,1111,Weighted Pullup,3,"[5, 5, 5]"
1870,1111,Standing Barbell Military Press (Front),3,"[8, 8, 8]"
1871,1112,Foam Roll - Rhomboids/Back,1,[0]


### TESTING + TRANSFORMATION FUNCTIONS

In [ ]:
def convert_output_to_list(model_outputs_str,model_name):
  model_outputs_list=[]
  if "gpt2" in model_name:
    for i,generated in enumerate(model_outputs):
      s = generated

      start_index = s.find("### Response")
      if start_index != -1:
        s = s[start_index:]

      s = s.replace("### Response:", "").strip()

      end_index = s.find(']}]') #POST PROCESSING
      if end_index != -1:
        s = s[:end_index + 3]

      try:
          s = ast.literal_eval(s)

      except (ValueError, SyntaxError):
          print(f"ERROR: Output at index {i} is invalid and cannot be parsed.")

      if type(s) != list or not all(isinstance(item, dict) for item in s):
        print(i,s,"ERROR: not dict or not list")

      else:
        model_outputs_list.append(s)


  elif "t5" in model_name:
    for i,generated in enumerate(model_outputs):
      s = generated
      s = s.replace(tokenizer.eos_token, "").strip()
      s = s.strip("<pad> ### Response: ")
      try:
          s = ast.literal_eval(s)

      except (ValueError, SyntaxError):  #POST PROCESSING
        last_bracket_index = s.rfind('}')
        if last_bracket_index != -1:
            s = s[:last_bracket_index + 1]
            s += ']'
            try:
                s=ast.literal_eval(s)
            except (ValueError, SyntaxError):
                print(f"ERROR: Output at index {i} is invalid and cannot be parsed.")

      if type(s) != list or not all(isinstance(item, dict) for item in s):
        print(i,s,"ERROR: not dict or not list")

      else:
        model_outputs_list.append(s)
  else:
    return "ERROR: Get Model Name"


  print(f"{len(model_outputs_list)} outputs converted correctly to list of dicts out of {len(model_outputs_str)} model outputs")
  return model_outputs_list


In [ ]:
def check_parameters_correctness(model_outputs_list): #MODIFY WHEN ADD PARMETERS OTHER THAN REPS (MUST HAVE EXERCISE,SETS AND AT LEAST ONE OF (reps,time,etc.)
  required_keys = {'exercise', 'sets', 'reps'} #(MODIFY)
  for i, outer_list in enumerate(model_outputs_list):
      for j, dictionary in enumerate(outer_list):
          if set(dictionary.keys()) != required_keys: #(MODIFY)
              print(f"ERROR: Dictionary at index [{i}][{j}] must have keys exactly {required_keys}.")
  print("All dictionaries have the correct keys.")



In [ ]:
def check_set_and_param_consistency(model_outputs_list): #MODIFY WHEN ADD PARMETERS OTHER THAN REPS (For ALL params other than exercise, #sets must be == len(param))
  c=0
  required_keys = {'exercise', 'sets', 'reps'}
  for i, outer_list in enumerate(model_outputs_list):
      for j, dictionary in enumerate(outer_list):
          if dictionary['sets']!=len(dictionary['reps']): #MODIFY
              c=c+1
              print(f"ERROR: Dictionary at index [{i}][{j}] has #sets != len(parameter)")
  print(f"inonsistency in sets and params for {c} out of {len(model_outputs_list)} ")


In [ ]:
def no_consecutive_same_exercise(model_outputs_list):
  c=0
  for i, outer_list in enumerate(model_outputs_list):
    k=1
    for j in range(0,len(outer_list)-1):
      if outer_list[j]==outer_list[j+1]:
        print(f"ERROR: Duplicated exercise at model output index {i}")
        if k==1:
          c=c+1
          k=0

  print(f"same exercise repeated consecutively for {c} out of {len(model_outputs_list)} ")



In [ ]:
def model_output_list_to_df(model_outputs_list):
  flat_data = [(i, d['exercise'], d['sets'], d['reps']) for i, outer_list in enumerate(model_outputs_list) for d in outer_list]
  df = pd.DataFrame(flat_data, columns=['block', 'exercise', 'sets', 'reps'])
  return df

,block,exercise,sets,reps
0,0,Hex Bar Deadlift,3,"[10, 10, 10]"
1,0,SB Crunch,3,"[20, 20, 20]"
2,0,Front Bridge,3,"[0, 0, 0]"
3,0,Back Bridge,3,"[10, 10, 0]"
4,1,Banded Clamshell,2,"[10, 10]"
...,...,...,...,...
1868,1111,Standing Barbell Military Press (Front),1,[8]
1869,1111,Weighted Pullup,3,"[5, 5, 5]"
1870,1111,Standing Barbell Military Press (Front),3,"[8, 8, 8]"
1871,1112,Foam Roll - Rhomboids/Back,1,[0]


### GPT2 RUN TESTS + TRANSFORMATION

In [ ]:
model_outputs_list=convert_output_to_list(model_outputs,model_name)

1113 outputs converted correctly to list of dicts out of 1113 model outputs


In [ ]:
check_parameters_correctness(model_outputs_list)

All dictionaries have the correct keys.


In [ ]:
check_set_and_param_consistency(model_outputs_list)

ERROR: Dictionary at index [217][0] has #sets != len(parameter)
ERROR: Dictionary at index [729][0] has #sets != len(parameter)
ERROR: Dictionary at index [1025][0] has #sets != len(parameter)
inonsistency in sets and params for 3 out of 1113 


In [ ]:
no_consecutive_same_exercise(model_outputs_list)

same exercise repeated consecutively for 0 out of 1113 


### GPT2 DATA ANALYSIS

In [ ]:
gpt2df=model_output_list_to_df(model_outputs_list)
gpt2df

,block,exercise,sets,reps
0,0,Barbell Hang Clean,4,"[3, 3, 3, 3]"
1,1,TRX Y,3,"[10, 10, 10]"
2,2,Foam Roll - Quads,1,[0]
3,2,Foam Roll - IT Band,1,[0]
4,3,Lateral Lunge,1,[8]
...,...,...,...,...
1226,1109,DB Bench Press,3,"[8, 8, 8]"
1227,1110,Barbell Bench Press,6,"[5, 5, 5, 5, 5, 5]"
1228,1111,Standing Barbell Military Press (Front),5,"[3, 3, 3, 3, 3]"
1229,1112,Foam Roll - Rhomboids/Back,1,[0]


In [ ]:
#DATA ANALYSIS ON EXERCISES
print("##############GPT2 OUTPUT##############")
total_number_of_ex = gpt2df['exercise'].count()
max_number_of_ex_per_block = gpt2df.groupby('block')['exercise'].count().max()
average_number_of_ex_per_outer_list = gpt2df.groupby('block')['exercise'].count().mean()
median_number_of_ex_per_outer_list = gpt2df.groupby('block')['exercise'].count().median()
std_dev_of_ex_per_outer_list = gpt2df.groupby('block')['exercise'].count().std()

print(f"Total number of exercises: {total_number_of_ex}")
print(f"Max number of exercises per block: {max_number_of_ex_per_block}")
print(f"Average number of exercises per block: {average_number_of_ex_per_outer_list}")
print(f"Median number of exercises per block: {median_number_of_ex_per_outer_list}")
print(f"Standard deviation of exercises per block: {std_dev_of_ex_per_outer_list}")

print("\n##############TRAINING DATASET OUTPUT##############")
total_number_of_ex = main_dataset_df['exercise'].count()
max_number_of_ex_per_block = main_dataset_df.groupby('block')['exercise'].count().max()
average_number_of_ex_per_outer_list = main_dataset_df.groupby('block')['exercise'].count().mean()
median_number_of_ex_per_outer_list = main_dataset_df.groupby('block')['exercise'].count().median()
std_dev_of_ex_per_outer_list = main_dataset_df.groupby('block')['exercise'].count().std()

print(f"Total number of exercises: {total_number_of_ex}")
print(f"Max number of exercises per block: {max_number_of_ex_per_block}")
print(f"Average number of exercises per block: {average_number_of_ex_per_outer_list}")
print(f"Median number of exercises per block: {median_number_of_ex_per_outer_list}")
print(f"Standard deviation of exercises per block: {std_dev_of_ex_per_outer_list}")

##############GPT2 OUTPUT##############
Total number of exercises: 1231
Max number of exercises per block: 3
Average number of exercises per block: 1.106019766397125
Median number of exercises per block: 1.0
Standard deviation of exercises per block: 0.35417558996887616

##############TRAINING DATASET OUTPUT##############
Total number of exercises: 1873
Max number of exercises per block: 5
Average number of exercises per block: 1.682839173405211
Median number of exercises per block: 1.0
Standard deviation of exercises per block: 1.0392660613307108


In [ ]:
#DATA ANALYSIS ON SETS
print("##############GPT2 OUTPUT##############")
total_sets = gpt2df['sets'].sum()
max_sets_per_block = gpt2df.groupby('block')['sets'].sum().max()
average_sets_per_outer_list = gpt2df.groupby('block')['sets'].sum().mean()
median_sets_per_outer_list = gpt2df.groupby('block')['sets'].sum().median()
std_dev_sets_per_outer_list = gpt2df.groupby('block')['sets'].sum().std()

print(f"Total number of sets: {total_sets}")
print(f"Max number of sets per block: {max_sets_per_block}")
print(f"Average number of sets per block: {average_sets_per_outer_list}")
print(f"Median number of sets per block: {median_sets_per_outer_list}")
print(f"Standard deviation of sets per block: {std_dev_sets_per_outer_list}")

print("\n##############TRAINING DATASET OUTPUT##############")
total_sets = main_dataset_df['sets'].sum()
max_sets_per_block = main_dataset_df.groupby('block')['sets'].sum().max()
average_sets_per_outer_list = main_dataset_df.groupby('block')['sets'].sum().mean()
median_sets_per_outer_list = main_dataset_df.groupby('block')['sets'].sum().median()
std_dev_sets_per_outer_list = main_dataset_df.groupby('block')['sets'].sum().std()

print(f"Total number of sets: {total_sets}")
print(f"Max number of sets per block: {max_sets_per_block}")
print(f"Average number of sets per block: {average_sets_per_outer_list}")
print(f"Median number of sets per block: {median_sets_per_outer_list}")
print(f"Standard deviation of sets per block: {std_dev_sets_per_outer_list}")

##############GPT2 OUTPUT##############
Total number of sets: 4048
Max number of sets per block: 11
Average number of sets per block: 3.637017070979335
Median number of sets per block: 3.0
Standard deviation of sets per block: 1.650527193049569

##############TRAINING DATASET OUTPUT##############
Total number of sets: 5773
Max number of sets per block: 24
Average number of sets per block: 5.186882300089847
Median number of sets per block: 4.0
Standard deviation of sets per block: 3.654002496718682


In [ ]:
#DATA ANALYSIS ON REPS
print("##############GPT2 OUTPUT##############")
gpt2df['max_reps'] = gpt2df['reps'].apply(lambda x: max(x) if x else 0)
gpt2df['mean_reps'] = gpt2df['reps'].apply(lambda x: sum(x) / len(x) if x else 0)
gpt2df['total_reps'] = gpt2df['reps'].apply(sum)

mean_reps = gpt2df['mean_reps'].mean()
median_reps = gpt2df['mean_reps'].median()
total_reps = gpt2df['total_reps'].sum()
max_reps_per_block = gpt2df.groupby('block')['total_reps'].sum().max()
average_reps_per_outer_list = gpt2df.groupby('block')['total_reps'].sum().mean()
median_reps_per_outer_list = gpt2df.groupby('block')['total_reps'].sum().median()
std_dev_reps_per_outer_list = gpt2df.groupby('block')['total_reps'].sum().std()

print(f"Mean number of average number of reps per exercise: {mean_reps}")
print(f"Median number of average number of reps per exercise: {median_reps}")
print(f"Total number of reps: {total_reps}")
print(f"Max number of reps per block: {max_reps_per_block}")
print(f"Average number of reps per block: {average_reps_per_outer_list}")
print(f"Median number of reps per block: {median_reps_per_outer_list}")
print(f"Standard deviation of reps per block: {std_dev_reps_per_outer_list}")

print("\n##############TRAINING DATASET OUTPUT##############")

main_dataset_df['max_reps'] = main_dataset_df['reps'].apply(lambda x: max(x) if x else 0)
main_dataset_df['mean_reps'] = main_dataset_df['reps'].apply(lambda x: sum(x) / len(x) if x else 0)
main_dataset_df['total_reps'] = main_dataset_df['reps'].apply(sum)

mean_reps = main_dataset_df['mean_reps'].mean()
median_reps = main_dataset_df['mean_reps'].median()
total_reps = main_dataset_df['total_reps'].sum()
max_reps_per_block = main_dataset_df.groupby('block')['total_reps'].sum().max()
average_reps_per_outer_list = main_dataset_df.groupby('block')['total_reps'].sum().mean()
median_reps_per_outer_list = main_dataset_df.groupby('block')['total_reps'].sum().median()
std_dev_reps_per_outer_list = main_dataset_df.groupby('block')['total_reps'].sum().std()

print(f"Mean number of average number of reps per exercise: {mean_reps}")
print(f"Median number of average number of reps per exercise: {median_reps}")
print(f"Total number of reps: {total_reps}")
print(f"Max number of reps per block: {max_reps_per_block}")
print(f"Average number of reps per block: {average_reps_per_outer_list}")
print(f"Median number of reps per block: {median_reps_per_outer_list}")
print(f"Standard deviation of reps per block: {std_dev_reps_per_outer_list}")

##############GPT2 OUTPUT##############
Mean number of average number of reps per exercise: 5.6569571776720435
Median number of average number of reps per exercise: 5.0
Total number of reps: 22011
Max number of reps per block: 70
Average number of reps per block: 19.776280323450134
Median number of reps per block: 20.0
Standard deviation of reps per block: 10.394177854638917

##############TRAINING DATASET OUTPUT##############
Mean number of average number of reps per exercise: 6.227431456542509
Median number of average number of reps per exercise: 5.0
Total number of reps: 32713
Max number of reps per block: 150
Average number of reps per block: 29.39173405211141
Median number of reps per block: 24.0
Standard deviation of reps per block: 24.454023884233184


### T5 RUN TESTS

In [ ]:
model_outputs_list=convert_output_to_list(model_outputs,model_name)

1113 outputs converted correctly to list of dicts out of 1113 model outputs


In [ ]:
check_parameters_correctness(model_outputs_list)

All dictionaries have the correct keys.


In [ ]:
check_set_and_param_consistency(model_outputs_list)

ERROR: Dictionary at index [14][0] has #sets != len(parameter)
ERROR: Dictionary at index [27][3] has #sets != len(parameter)
ERROR: Dictionary at index [53][0] has #sets != len(parameter)
ERROR: Dictionary at index [82][0] has #sets != len(parameter)
ERROR: Dictionary at index [82][1] has #sets != len(parameter)
ERROR: Dictionary at index [107][0] has #sets != len(parameter)
ERROR: Dictionary at index [190][0] has #sets != len(parameter)
ERROR: Dictionary at index [190][1] has #sets != len(parameter)
ERROR: Dictionary at index [217][2] has #sets != len(parameter)
ERROR: Dictionary at index [218][0] has #sets != len(parameter)
ERROR: Dictionary at index [218][1] has #sets != len(parameter)
ERROR: Dictionary at index [235][2] has #sets != len(parameter)
ERROR: Dictionary at index [249][0] has #sets != len(parameter)
ERROR: Dictionary at index [249][1] has #sets != len(parameter)
ERROR: Dictionary at index [266][0] has #sets != len(parameter)
ERROR: Dictionary at index [267][0] has #sets

In [ ]:
no_consecutive_same_exercise(model_outputs_list)

ERROR: Duplicated exercise at model output index 0
ERROR: Duplicated exercise at model output index 27
ERROR: Duplicated exercise at model output index 29
ERROR: Duplicated exercise at model output index 43
ERROR: Duplicated exercise at model output index 50
ERROR: Duplicated exercise at model output index 50
ERROR: Duplicated exercise at model output index 56
ERROR: Duplicated exercise at model output index 56
ERROR: Duplicated exercise at model output index 62
ERROR: Duplicated exercise at model output index 73
ERROR: Duplicated exercise at model output index 78
ERROR: Duplicated exercise at model output index 78
ERROR: Duplicated exercise at model output index 81
ERROR: Duplicated exercise at model output index 83
ERROR: Duplicated exercise at model output index 93
ERROR: Duplicated exercise at model output index 94
ERROR: Duplicated exercise at model output index 98
ERROR: Duplicated exercise at model output index 100
ERROR: Duplicated exercise at model output index 101
ERROR: Dupl

### T5 DATA ANALYSIS

In [ ]:
t5df=model_output_list_to_df(model_outputs_list)
t5df

,block,exercise,sets,reps
0,0,Barbell Clean Pull,3,"[10, 10, 10]"
1,0,SB Pot Stirrer,3,"[10, 10, 10]"
2,0,SB Pot Stirrer,3,"[10, 10, 10]"
3,1,TRX Hamstring Curl,3,"[10, 10, 10]"
4,2,Side Bridge - Hip Raise,2,"[10, 10]"
...,...,...,...,...
1589,1108,Barbell Deadlift,4,"[5, 5, 5, 5]"
1590,1109,DB Bench Press,4,"[5, 5, 5, 5]"
1591,1110,Barbell Bench Press,6,"[5, 5, 5, 5, 5, 5]"
1592,1111,Barbell Bench Press,5,"[5, 5, 5, 5, 5]"


In [ ]:
#DATA ANALYSIS ON EXERCISES
print("##############T5 OUTPUT##############")
total_number_of_ex = t5df['exercise'].count()
max_number_of_ex_per_block = t5df.groupby('block')['exercise'].count().max()
average_number_of_ex_per_outer_list = t5df.groupby('block')['exercise'].count().mean()
median_number_of_ex_per_outer_list = t5df.groupby('block')['exercise'].count().median()
std_dev_of_ex_per_outer_list = t5df.groupby('block')['exercise'].count().std()

print(f"Total number of exercises: {total_number_of_ex}")
print(f"Max number of exercises per block: {max_number_of_ex_per_block}")
print(f"Average number of exercises per block: {average_number_of_ex_per_outer_list}")
print(f"Median number of exercises per block: {median_number_of_ex_per_outer_list}")
print(f"Standard deviation of exercises per block: {std_dev_of_ex_per_outer_list}")

print("\n##############TRAINING DATASET OUTPUT##############")
total_number_of_ex = main_dataset_df['exercise'].count()
max_number_of_ex_per_block = main_dataset_df.groupby('block')['exercise'].count().max()
average_number_of_ex_per_outer_list = main_dataset_df.groupby('block')['exercise'].count().mean()
median_number_of_ex_per_outer_list = main_dataset_df.groupby('block')['exercise'].count().median()
std_dev_of_ex_per_outer_list = main_dataset_df.groupby('block')['exercise'].count().std()

print(f"Total number of exercises: {total_number_of_ex}")
print(f"Max number of exercises per block: {max_number_of_ex_per_block}")
print(f"Average number of exercises per block: {average_number_of_ex_per_outer_list}")
print(f"Median number of exercises per block: {median_number_of_ex_per_outer_list}")
print(f"Standard deviation of exercises per block: {std_dev_of_ex_per_outer_list}")

##############T5 OUTPUT##############
Total number of exercises: 1594
Max number of exercises per block: 5
Average number of exercises per block: 1.4321653189577719
Median number of exercises per block: 1.0
Standard deviation of exercises per block: 0.787222876660194

##############TRAINING DATASET OUTPUT##############
Total number of exercises: 1873
Max number of exercises per block: 5
Average number of exercises per block: 1.682839173405211
Median number of exercises per block: 1.0
Standard deviation of exercises per block: 1.0392660613307108


In [ ]:
#DATA ANALYSIS ON SETS
print("##############T5 OUTPUT##############")
total_sets = t5df['sets'].sum()
max_sets_per_block = t5df.groupby('block')['sets'].sum().max()
average_sets_per_outer_list = t5df.groupby('block')['sets'].sum().mean()
median_sets_per_outer_list = t5df.groupby('block')['sets'].sum().median()
std_dev_sets_per_outer_list = t5df.groupby('block')['sets'].sum().std()

print(f"Total number of sets: {total_sets}")
print(f"Max number of sets per block: {max_sets_per_block}")
print(f"Average number of sets per block: {average_sets_per_outer_list}")
print(f"Median number of sets per block: {median_sets_per_outer_list}")
print(f"Standard deviation of sets per block: {std_dev_sets_per_outer_list}")

print("\n##############TRAINING DATASET OUTPUT##############")
total_sets = main_dataset_df['sets'].sum()
max_sets_per_block = main_dataset_df.groupby('block')['sets'].sum().max()
average_sets_per_outer_list = main_dataset_df.groupby('block')['sets'].sum().mean()
median_sets_per_outer_list = main_dataset_df.groupby('block')['sets'].sum().median()
std_dev_sets_per_outer_list = main_dataset_df.groupby('block')['sets'].sum().std()

print(f"Total number of sets: {total_sets}")
print(f"Max number of sets per block: {max_sets_per_block}")
print(f"Average number of sets per block: {average_sets_per_outer_list}")
print(f"Median number of sets per block: {median_sets_per_outer_list}")
print(f"Standard deviation of sets per block: {std_dev_sets_per_outer_list}")

##############T5 OUTPUT##############
Total number of sets: 4973
Max number of sets per block: 17
Average number of sets per block: 4.468104222821204
Median number of sets per block: 3.0
Standard deviation of sets per block: 2.857389401283396

##############TRAINING DATASET OUTPUT##############
Total number of sets: 5773
Max number of sets per block: 24
Average number of sets per block: 5.186882300089847
Median number of sets per block: 4.0
Standard deviation of sets per block: 3.654002496718682


In [ ]:
#DATA ANALYSIS ON REPS
print("##############T5 OUTPUT##############")
t5df['max_reps'] = t5df['reps'].apply(lambda x: max(x) if x else 0)
t5df['mean_reps'] = t5df['reps'].apply(lambda x: sum(x) / len(x) if x else 0)
t5df['total_reps'] = t5df['reps'].apply(sum)

mean_reps = t5df['mean_reps'].mean()
median_reps = t5df['mean_reps'].median()
total_reps = t5df['total_reps'].sum()
max_reps_per_block = t5df.groupby('block')['total_reps'].sum().max()
average_reps_per_outer_list = t5df.groupby('block')['total_reps'].sum().mean()
median_reps_per_outer_list = t5df.groupby('block')['total_reps'].sum().median()
std_dev_reps_per_outer_list = t5df.groupby('block')['total_reps'].sum().std()

print(f"Mean number of average number of reps per exercise: {mean_reps}")
print(f"Median number of average number of reps per exercise: {median_reps}")
print(f"Total number of reps: {total_reps}")
print(f"Max number of reps per block: {max_reps_per_block}")
print(f"Average number of reps per block: {average_reps_per_outer_list}")
print(f"Median number of reps per block: {median_reps_per_outer_list}")
print(f"Standard deviation of reps per block: {std_dev_reps_per_outer_list}")

print("\n##############TRAINING DATASET OUTPUT##############")

main_dataset_df['max_reps'] = main_dataset_df['reps'].apply(lambda x: max(x) if x else 0)
main_dataset_df['mean_reps'] = main_dataset_df['reps'].apply(lambda x: sum(x) / len(x) if x else 0)
main_dataset_df['total_reps'] = main_dataset_df['reps'].apply(sum)

mean_reps = main_dataset_df['mean_reps'].mean()
median_reps = main_dataset_df['mean_reps'].median()
total_reps = main_dataset_df['total_reps'].sum()
max_reps_per_block = main_dataset_df.groupby('block')['total_reps'].sum().max()
average_reps_per_outer_list = main_dataset_df.groupby('block')['total_reps'].sum().mean()
median_reps_per_outer_list = main_dataset_df.groupby('block')['total_reps'].sum().median()
std_dev_reps_per_outer_list = main_dataset_df.groupby('block')['total_reps'].sum().std()

print(f"Mean number of average number of reps per exercise: {mean_reps}")
print(f"Median number of average number of reps per exercise: {median_reps}")
print(f"Total number of reps: {total_reps}")
print(f"Max number of reps per block: {max_reps_per_block}")
print(f"Average number of reps per block: {average_reps_per_outer_list}")
print(f"Median number of reps per block: {median_reps_per_outer_list}")
print(f"Standard deviation of reps per block: {std_dev_reps_per_outer_list}")

##############T5 OUTPUT##############
Mean number of average number of reps per exercise: 6.3579553601981456
Median number of average number of reps per exercise: 5.0
Total number of reps: 29678
Max number of reps per block: 120
Average number of reps per block: 26.664869721473494
Median number of reps per block: 24.0
Standard deviation of reps per block: 18.539865573221405

##############TRAINING DATASET OUTPUT##############
Mean number of average number of reps per exercise: 6.227431456542509
Median number of average number of reps per exercise: 5.0
Total number of reps: 32713
Max number of reps per block: 150
Average number of reps per block: 29.39173405211141
Median number of reps per block: 24.0
Standard deviation of reps per block: 24.454023884233184


# SAVING THE MODEL

In [ ]:
model.save_pretrained(dir+sub_dir+'final_model_'+original_model_name)
tokenizer.save_pretrained(dir+sub_dir+'final_model_'+original_model_name)


('/content/drive/MyDrive/BridgeAthletics//Training/final_model_t5-base/tokenizer_config.json',
 '/content/drive/MyDrive/BridgeAthletics//Training/final_model_t5-base/special_tokens_map.json',
 '/content/drive/MyDrive/BridgeAthletics//Training/final_model_t5-base/spiece.model',
 '/content/drive/MyDrive/BridgeAthletics//Training/final_model_t5-base/added_tokens.json')